# 04 — Retrieval test (query → retrieved regulation)

**Purpose:** Load the persisted ChromaDB index, run a query, display retrieved regulation text and metadata. No LLM — validation of RAG-ready retrieval only.

**Input:** Query string (edit below). **Output:** Top-k chunks with `text`, `source`, `country`, `category`.

In [3]:
# Avoid TensorFlow/tensorboard load (same as notebook 03). If you see MessageFactory/GetPrototype error: pip install 'protobuf>=3.20,<4'
import os
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [4]:
from pathlib import Path
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
INDEX_DIR = PROJECT_ROOT / "outputs" / "vector_index"

if not INDEX_DIR.exists():
    raise FileNotFoundError(f"Vector index not found. Run notebook 03 first. Expected: {INDEX_DIR}")

client = chromadb.PersistentClient(path=str(INDEX_DIR), settings=Settings(anonymized_telemetry=False))
collection = client.get_collection("policy_chunks")
count = collection.count()
if count == 0:
    raise ValueError("Collection 'policy_chunks' is empty. Run notebook 03 to build the index.")

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Collection count: {count}")

Collection count: 649


In [5]:
def retrieve(query: str, top_k: int = 5):
    q_emb = model.encode([query])
    n = min(top_k, collection.count())
    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=n,
        include=["documents", "metadatas"],
    )
    return results


# --- Edit your query here ---
query = "When must adverse events be reported to the regulator?"
top_k = 5

results = retrieve(query, top_k)
# ChromaDB returns lists per query; handle None or missing keys
docs = (results.get("documents") or [[]])[0] or []
metas = (results.get("metadatas") or [[]])[0] or []

print(f"Query: {query}\n")
print("=" * 60)
for i, (doc, meta) in enumerate(zip(docs, metas), 1):
    meta = meta or {}
    source = meta.get("source", "")
    country = meta.get("country", "")
    category = meta.get("category", "")
    text = (doc or "")[:800]
    if len(doc or "") > 800:
        text += "..."
    print(f"\n--- Result {i} | {source} | {country} | {category} ---\n")
    print(text)
    print()
if not docs:
    print("No results returned. Check that notebook 03 has been run and the index contains chunks.")

Query: When must adverse events be reported to the regulator?


--- Result 1 | EMA | EU | Pharmacovigilance ---

Article 27(3) of Regulation (EC) No 726/2004;  it is included in VI. Appendix
2.  
The electronic reporting recommendation s regarding suspected adverse reactions reports published in 
the scientific and medical  literature are provided in VI.C.6.2.3.2 .. 
VI.C.2.2.


--- Result 2 | EMA | EU | Pharmacovigilance ---

1. Causality  
In accordance with ICH -E2A (see GVP Annex IV) , the definition of an adverse reaction implies at least 
a reasonable possibility of a causal relation ship between a suspected medicinal product and an adverse 
event. A n adverse  reaction, in contrast to an adverse event, is characterised by the fact that a causal 
relationship between a  medicinal product  and an occurrence is suspected. For regulatory report ing 
purposes, as detailed in ICH -E2D (see GVP Annex IV) , if an event is spontaneously reported, even if 
the relationship is unknown or u